In [1]:

import torch
from pathlib import Path
import sys

In [2]:

LCL_PATH  = str(Path().cwd())
ROOT_PATH = str(Path(LCL_PATH).parent.parent.parent.parent)
DEEPL_PATH = str(Path(ROOT_PATH)/"deep_learning")
print("""
root path:\t{}
local path:\t{}
deep learning path:\t{}""".format(ROOT_PATH, LCL_PATH, DEEPL_PATH))


root path:	/home/gheorghe/Desktop/Proiecte/master/CapitoleAvansateDinReteleNeuronale/CARN
local path:	/home/gheorghe/Desktop/Proiecte/master/CapitoleAvansateDinReteleNeuronale/CARN/deep_learning/layers/test/unet_blocks
deep learning path:	/home/gheorghe/Desktop/Proiecte/master/CapitoleAvansateDinReteleNeuronale/CARN/deep_learning


In [3]:

# adding local_folder to the system path
sys.path.append(ROOT_PATH)
sys.path.append(LCL_PATH)
sys.path.append(DEEPL_PATH)

from sys_function import * # este in root

In [5]:

sys_remove_modules("layers.unet_blocks.unet_resnet_block")

from layers.unet_blocks.unet_resnet_block import *


# Build model

In [6]:

unet_conf = dict(
    encode=dict(
        enc_conv_1x=dict(in_channels=32, 
                     expansion=4, 
                     stride=2, 
                     intermediate_channels=32, 
                     num_residual_blocks=2),
        enc_conv_2x=dict(in_channels=128, 
                     expansion=4, 
                     stride=2, 
                     intermediate_channels=64, 
                     num_residual_blocks=2),
        enc_conv_3x=dict(in_channels=256, 
                     expansion=4, 
                     stride=2, 
                     intermediate_channels=256, 
                     num_residual_blocks=2),
    ),
    decode=dict(
        dec_bottleneck=dict(in_channels=1024, 
                     expansion=4, 
                     stride=1, 
                     intermediate_channels=256, 
                     num_residual_blocks=2),
        dec_conv_3x=dict(in_channels=1024, 
                     expansion=4, 
                     stride=2, 
                     intermediate_channels=64, 
                     num_residual_blocks=2),
        dec_conv_2x=dict(in_channels=256, 
                     expansion=4, 
                     stride=2, 
                     intermediate_channels=32, 
                     num_residual_blocks=2),
        dec_conv_1x=dict(in_channels=128, 
                     expansion=4, 
                     stride=2, 
                     intermediate_channels=8, 
                     num_residual_blocks=2),
    ),
)

In [8]:

model = UNetResNetBlock("unet", **unet_conf)

In [9]:
model.encode.block

Sequential(
  (enc_conv_1x_0): IdentityResNet(
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv1): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv3): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (activ_fn): SiLU(inplace=True)
    (identity_downsample): IdentityConv2dDownSample(
      (conv1): Conv2d(32, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (enc_conv_1x_1): IdentityResNet(
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv1): Conv2d(128, 32, kernel_size=

In [10]:

enc_features = []
u_feature = torch.rand(1, 32, 32,  32)
enc_features.append(u_feature)
for layer in model.encode.block:
    u_feature = layer(u_feature)
    print("enc u_feature", u_feature.shape)
    enc_features.append(u_feature)
print()
print()
for layer, x_feature in zip(model.decode.block, enc_features[::-1]):
    #print(layer)
    print("dec x_feature:", x_feature.shape)
    print("dec u_feature before:", u_feature.shape)
    u_feature = layer(u_feature)
    print("dec u_feature after:", u_feature.shape)
    u_feature = u_feature + x_feature
    print()

enc u_feature torch.Size([1, 128, 16, 16])
enc u_feature torch.Size([1, 128, 16, 16])
enc u_feature torch.Size([1, 256, 8, 8])
enc u_feature torch.Size([1, 256, 8, 8])
enc u_feature torch.Size([1, 1024, 4, 4])
enc u_feature torch.Size([1, 1024, 4, 4])


dec x_feature: torch.Size([1, 1024, 4, 4])
dec u_feature before: torch.Size([1, 1024, 4, 4])
dec u_feature after: torch.Size([1, 1024, 4, 4])

dec x_feature: torch.Size([1, 1024, 4, 4])
dec u_feature before: torch.Size([1, 1024, 4, 4])
dec u_feature after: torch.Size([1, 1024, 4, 4])

dec x_feature: torch.Size([1, 256, 8, 8])
dec u_feature before: torch.Size([1, 1024, 4, 4])
dec u_feature after: torch.Size([1, 256, 8, 8])

dec x_feature: torch.Size([1, 256, 8, 8])
dec u_feature before: torch.Size([1, 256, 8, 8])
dec u_feature after: torch.Size([1, 256, 8, 8])

dec x_feature: torch.Size([1, 128, 16, 16])
dec u_feature before: torch.Size([1, 256, 8, 8])
dec u_feature after: torch.Size([1, 128, 16, 16])

dec x_feature: torch.Size([1, 128, 

In [11]:
model(u_feature).shape

torch.Size([1, 32, 32, 32])